In [ ]:
code = 'IRONFLY'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def IRONFLY(bt, start_time, end_time, sell_om, buy_om, sl):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)

        while True:

            sell_ce_scrip, sell_pe_scrip, _, _, future_price, sell_start_dt = bt.get_strike(start_dt, end_dt, om=sell_om)
            if sell_ce_scrip is None: return None

            buy_ce_scrip, buy_pe_scrip, _, _, _, buy_start_dt = bt.get_strike(sell_start_dt, end_dt, om=buy_om)
            if buy_ce_scrip is None: return None
            
            if sell_start_dt == buy_start_dt:
                break
            else:
                start_dt = sell_start_dt + datetime.timedelta(minutes=1)
            

        sell_ce_data, sell_pe_data = bt.get_straddle_data(start_dt, end_dt, sell_ce_scrip, sell_pe_scrip, seperate=True)
        buy_ce_data, buy_pe_data = bt.get_straddle_data(start_dt, end_dt, buy_ce_scrip, buy_pe_scrip, seperate=True)

        sell_ce_data = sell_ce_data[sell_ce_data['date_time'].isin(buy_ce_data['date_time'])]
        sell_pe_data = sell_pe_data[sell_pe_data['date_time'].isin(buy_pe_data['date_time'])]

        buy_ce_data = buy_ce_data[buy_ce_data['date_time'].isin(sell_ce_data['date_time'])]
        buy_pe_data = buy_pe_data[buy_pe_data['date_time'].isin(sell_pe_data['date_time'])]
        
        if sell_ce_data.empty or buy_ce_data.empty:
            return None        

        entry_time = sell_ce_data['date_time'].iloc[0]
        
        sell_ce_price = sell_ce_data['close'].iloc[0]
        sell_pe_price = sell_pe_data['close'].iloc[0]
        sell_std_premium = sell_ce_price + sell_pe_price
        sell_eod_premium = sell_ce_data['close'].iloc[-1] + sell_pe_data['close'].iloc[-1]
        
        buy_ce_price = buy_ce_data['close'].iloc[0]
        buy_pe_price = buy_pe_data['close'].iloc[0]
        buy_std_premium = buy_ce_price + buy_pe_price
        buy_eod_premium = buy_ce_data['close'].iloc[-1] + buy_pe_data['close'].iloc[-1]
        
        if sell_std_premium <= buy_std_premium:
            return None
        
        sell_straddle = [sell_ce_scrip, sell_ce_price, sell_pe_scrip, sell_pe_price, sell_std_premium, sell_eod_premium]
        buy_straddle = [buy_ce_scrip, buy_ce_price, buy_pe_scrip, buy_pe_price, buy_std_premium, buy_eod_premium]
        
        initial_premium = (sell_std_premium - buy_std_premium)
        eod_premium = (sell_eod_premium - buy_eod_premium)

        slipage = bt.Cal_slipage(sell_std_premium + buy_std_premium)
        no_sl_pnl = round((initial_premium - eod_premium) - slipage, 2)

        sl_hit, sl_time, pnl = False, '', 0
        sl_price = initial_premium * (1 + (sl/100))
        candle_close_price_list = ((sell_ce_data.close + sell_pe_data.close) - (buy_ce_data.close + buy_pe_data.close)).tolist()
        
        try:
            hit_value = [ele for ele in candle_close_price_list if ele >= sl_price][0]
            sl_hit = True
            sl_index = candle_close_price_list.index(hit_value)
            sl_time = sell_ce_data['date_time'].iloc[sl_index].time()
            sl_pnl = round((initial_premium - hit_value) - slipage, 2)
        except:
            sl_pnl = no_sl_pnl
            
        mtm_list = [initial_premium, eod_premium, no_sl_pnl, sl_hit, sl_time, sl_pnl]
        
        return [code, bt.index, start_time, end_time, sell_om, buy_om, sl, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time.time(), future_price] + sell_straddle + buy_straddle + mtm_list
    
    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, sell_om, buy_om, sl])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            max_re, re_entries = 10, 10

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_SellOM/P_BuyOM/P_SL/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future/CE.Scrip/CE.Price/PE.Scrip/PE.Price/STD.Premium/STD.EOD.Premium/CE.Wing/PE.Wing/CE.Wing.Price/PE.Wing.Price/Wing.Premium/Wing.EOD.Premium/Initial.Premium/EOD.Premium/NO.SL.PNL/SL.Hit/SL.Time/SL.PNL'.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [IRONFLY(wbt, row['entry_time'], row['exit_time'], row['sell_om'], row['buy_om'], row['sl']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))